### E2

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn import metrics

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsRegressor, KNeighborsClassifier

In [ ]:
df = pd.read_csv("E2_datos.csv")

Viendo el csv, hay hartas filas con "nulo", así que como supuesto vamos a decir que esas filas con "nulo" son valores faltantes, y las vamos a reemplazar por NaN para poder trabajar con ellas.

In [3]:
df = df.replace("nulo", np.nan)

Fijémonos en que las misiones 1 y 2 intentan predecir O3 y PM2.5 respectivamente, pero no se especifica qué hacer con las filas que tienen ambos valores faltantes a la vez. 

Podría haber un bloqueo circular, donde para predecir O3 necesitamos PM2.5 y para predecir PM2.5 necesitamos O3. Y si no están ambos, no podríamos predecir ninguno de los dos.

Para eso, vamos a crear una máscara para identificar esas filas y luego decidir qué hacer con ellas.

In [4]:
mask_bloqueo = (
    df["O3"].isna() &
    df["PM2.5"].isna() &
    df["Year"].notna() &
    df["Month"].notna() &
    df["Day"].notna()
)
print("\nFilas con O3 y PM2.5 faltantes a la vez:", mask_bloqueo.sum())


Filas con O3 y PM2.5 faltantes a la vez: 37


Las vamos a dejar fuera.

In [5]:
df = df[~mask_bloqueo].copy()
df

,Year,Month,Day,O3,PM2.5,Environmental_risk
0,2008,1,1,29.63,NaN,NaN
1,2008,1,2,21.46,NaN,NaN
2,2008,1,3,24.25,NaN,NaN
3,2008,1,4,29.04,NaN,NaN
4,2008,1,5,30.17,NaN,NaN
...,...,...,...,...,...,...
2979,2016,2,27,22.93,17.12,Bajo
2980,2016,2,28,26.70,15.33,Bajo
2981,2016,2,29,24.62,16.96,Bajo
2982,2016,3,1,28.86,32.67,medio


In [6]:
# por si acaso
mask_faltantes_fecha = (
    df["Year"].isna() |
    df["Month"].isna() |
    df["Day"].isna()
)
print("\nFilas con faltantes en Year, Month o Day:", mask_faltantes_fecha.sum())


Filas con faltantes en Year, Month o Day: 0


Además, son 37 de 2947 filas, así que no es un número tan grande como para preocuparnos por perder mucha información.

Vamos con la codificación

In [7]:
columnas_categoricas = ["Year", "Month", "Day"]

encoders = {}
for col in columnas_categoricas:
    le = LabelEncoder()
    df.loc[df[col].notna(), col] = le.fit_transform(df.loc[df[col].notna(), col])
    encoders[col] = le

### Misión 1: prediccion de variables numericas (PM2.5)

In [8]:
df_pm = df.dropna(subset=["Year", "Month", "Day", "O3", "PM2.5"]).copy()

features_pm = ["Year", "Month", "Day", "O3"]
target_pm = "PM2.5"

X_pm = df_pm[features_pm].copy()
y_pm = df_pm[[target_pm]].copy()


# train y test

X_train_pm, X_test_pm, y_train_pm, y_test_pm = train_test_split(
    X_pm,
    y_pm,
    test_size=0.2,
    random_state=42
)

# normalizamos
# solo las variables numéricas: O3 y PM2.5

scaler_X_pm = StandardScaler()
X_train_pm["O3"] = scaler_X_pm.fit_transform(X_train_pm[["O3"]])
X_test_pm["O3"] = scaler_X_pm.transform(X_test_pm[["O3"]])

scaler_y_pm = StandardScaler()
y_train_pm_scaled = scaler_y_pm.fit_transform(y_train_pm)
y_test_pm_scaled = scaler_y_pm.transform(y_test_pm)


Ahora ya son modelos de regresión y no clasificación, ojo con esto.

In [9]:
# Modelo 1: Regresión lineal
model_pm_1 = LinearRegression()
model_pm_1.fit(X_train_pm, y_train_pm_scaled.ravel())

pred_pm_1_scaled = model_pm_1.predict(X_test_pm).reshape(-1, 1)
pred_pm_1 = scaler_y_pm.inverse_transform(pred_pm_1_scaled).ravel()

mse_pm_1 = metrics.mean_squared_error(y_test_pm, pred_pm_1)
mae_pm_1 = metrics.mean_absolute_error(y_test_pm, pred_pm_1)
mape_pm_1 = metrics.mean_absolute_percentage_error(y_test_pm, pred_pm_1)


# Modelo 2: KNN Regressor

model_pm_2 = KNeighborsRegressor()
model_pm_2.fit(X_train_pm, y_train_pm_scaled.ravel())

pred_pm_2_scaled = model_pm_2.predict(X_test_pm).reshape(-1, 1)
pred_pm_2 = scaler_y_pm.inverse_transform(pred_pm_2_scaled).ravel()

mse_pm_2 = metrics.mean_squared_error(y_test_pm, pred_pm_2)
mae_pm_2 = metrics.mean_absolute_error(y_test_pm, pred_pm_2)
mape_pm_2 = metrics.mean_absolute_percentage_error(y_test_pm, pred_pm_2)

In [10]:
print(f"LinearRegression -> MSE: {mse_pm_1}, MAE: {mae_pm_1}, MAPE: {mape_pm_1}")
print(f"KNeighborsRegressor -> MSE: {mse_pm_2}, MAE: {mae_pm_2}, MAPE: {mape_pm_2}")

# elegir mejor modelo según MSE, que penaliza más los errores grandes, 
# lo cual es importante en este caso para evitar predicciones muy alejadas de la realidad
if mse_pm_1 <= mse_pm_2:
    mejor_modelo_pm = model_pm_1
    mejor_nombre_pm = "LinearRegression"
    print("Mejor modelo para PM2.5:", mejor_nombre_pm)
else:
    mejor_modelo_pm = model_pm_2
    mejor_nombre_pm = "KNeighborsRegressor"
    print("Mejor modelo para PM2.5:", mejor_nombre_pm)


LinearRegression -> MSE: 226.3639864632186, MAE: 11.341606468690422, MAPE: 0.5508086747857327
KNeighborsRegressor -> MSE: 131.56984733834585, MAE: 8.151278195488722, MAPE: 0.36671207359513835
Mejor modelo para PM2.5: KNeighborsRegressor


Ahora ya podemos imputar valores.

In [11]:
mask_fill_pm = (
    df["PM2.5"].isna() &
    df["Year"].notna() &
    df["Month"].notna() &
    df["Day"].notna() &
    df["O3"].notna()
)

if mask_fill_pm.sum() > 0:
    X_fill_pm = df.loc[mask_fill_pm, features_pm].copy()

    # normalizar O3 con el scaler ajustado en train
    X_fill_pm["O3"] = scaler_X_pm.transform(X_fill_pm[["O3"]])

    # predecir en escala normalizada del target
    pred_fill_pm_scaled = mejor_modelo_pm.predict(X_fill_pm).reshape(-1, 1)

    # volver a escala original de PM2.5
    pred_fill_pm = scaler_y_pm.inverse_transform(pred_fill_pm_scaled).ravel()

    df.loc[mask_fill_pm, "PM2.5"] = pred_fill_pm
    
print("\nFaltantes en PM2.5 después de imputación:", df["PM2.5"].isna().sum())


Faltantes en PM2.5 después de imputación: 0


### Mision 2: prediccion de variables numericas parte (O3)

In [12]:
# usar solo filas completas para las columnas requeridas
df_o3 = df.dropna(subset=["Year", "Month", "Day", "PM2.5", "O3"]).copy()

features_o3 = ["Year", "Month", "Day", "PM2.5"]
target_o3 = "O3"

X_o3 = df_o3[features_o3].copy()
y_o3 = df_o3[[target_o3]].copy()

X_train_o3, X_test_o3, y_train_o3, y_test_o3 = train_test_split(
    X_o3,
    y_o3,
    test_size=0.2,
    random_state=42
)

# normalizamos
# solo las variables numéricas: PM2.5 y O3

scaler_X_o3 = StandardScaler()
X_train_o3["PM2.5"] = scaler_X_o3.fit_transform(X_train_o3[["PM2.5"]])
X_test_o3["PM2.5"] = scaler_X_o3.transform(X_test_o3[["PM2.5"]])

scaler_y_o3 = StandardScaler()
y_train_o3_scaled = scaler_y_o3.fit_transform(y_train_o3)
y_test_o3_scaled = scaler_y_o3.transform(y_test_o3)


# Modelo 1: Regresión lineal

model_o3_1 = LinearRegression()
model_o3_1.fit(X_train_o3, y_train_o3_scaled.ravel())

pred_o3_1_scaled = model_o3_1.predict(X_test_o3).reshape(-1, 1)
pred_o3_1 = scaler_y_o3.inverse_transform(pred_o3_1_scaled).ravel()

mse_o3_1 = metrics.mean_squared_error(y_test_o3, pred_o3_1)
mae_o3_1 = metrics.mean_absolute_error(y_test_o3, pred_o3_1)
mape_o3_1 = metrics.mean_absolute_percentage_error(y_test_o3, pred_o3_1)


# Modelo 2: KNN Regressor

model_o3_2 = KNeighborsRegressor()
model_o3_2.fit(X_train_o3, y_train_o3_scaled.ravel())

pred_o3_2_scaled = model_o3_2.predict(X_test_o3).reshape(-1, 1)
pred_o3_2 = scaler_y_o3.inverse_transform(pred_o3_2_scaled).ravel()

mse_o3_2 = metrics.mean_squared_error(y_test_o3, pred_o3_2)
mae_o3_2 = metrics.mean_absolute_error(y_test_o3, pred_o3_2)
mape_o3_2 = metrics.mean_absolute_percentage_error(y_test_o3, pred_o3_2)


In [13]:
# RESULTADOS

print("\nMISIÓN 2 - Predicción de O3")
print("LinearRegression -> MSE:", mse_o3_1, "| MAE:", mae_o3_1, "| MAPE:", mape_o3_1)
print("KNeighborsRegressor -> MSE:", mse_o3_2, "| MAE:", mae_o3_2, "| MAPE:", mape_o3_2)

# elegir mejor modelo según MSE
if mse_o3_1 <= mse_o3_2:
    mejor_modelo_o3 = model_o3_1
    mejor_nombre_o3 = "LinearRegression"
    print("Mejor modelo para O3:", mejor_nombre_o3)
else:
    mejor_modelo_o3 = model_o3_2
    mejor_nombre_o3 = "KNeighborsRegressor"
    print("Mejor modelo para O3:", mejor_nombre_o3)



MISIÓN 2 - Predicción de O3
LinearRegression -> MSE: 45.471183942999495 | MAE: 5.509976275375457 | MAPE: 0.5954401683898811
KNeighborsRegressor -> MSE: 18.70152084027778 | MAE: 3.3694340277777775 | MAPE: 0.31955631839583504
Mejor modelo para O3: KNeighborsRegressor


In [14]:
mask_fill_o3 = (
    df["O3"].isna() &
    df["Year"].notna() &
    df["Month"].notna() &
    df["Day"].notna() &
    df["PM2.5"].notna()
)

if mask_fill_o3.sum() > 0:
    X_fill_o3 = df.loc[mask_fill_o3, features_o3].copy()

    # normalizar PM2.5 con el scaler ajustado en train
    X_fill_o3["PM2.5"] = scaler_X_o3.transform(X_fill_o3[["PM2.5"]])

    # predecir en escala normalizada del target
    pred_fill_o3_scaled = mejor_modelo_o3.predict(X_fill_o3).reshape(-1, 1)

    # volver a escala original de O3
    pred_fill_o3 = scaler_y_o3.inverse_transform(pred_fill_o3_scaled).ravel()

    df.loc[mask_fill_o3, "O3"] = pred_fill_o3

# ver nans
print("\nVALORES FALTANTES DESPUÉS DE MISIONES 1 Y 2")
print(df.isna().sum())


VALORES FALTANTES DESPUÉS DE MISIONES 1 Y 2
Year                    0
Month                   0
Day                     0
O3                      0
PM2.5                   0
Environmental_risk    425
dtype: int64


### Mision 3: prediccion de variables categoricas (Environmental_risk)

In [15]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import balanced_accuracy_score


df_clf = df.dropna(subset=["Year", "Month", "Day", "O3", "PM2.5", "Environmental_risk"]).copy()

features_clf = ["Year", "Month", "Day", "O3", "PM2.5"]
target_clf = "Environmental_risk"

X_clf = df_clf[features_clf].copy()
y_clf = df_clf[target_clf].copy()

Codificamos la variable Environmental_risk, que es la variable objetivo de esta misión.

In [16]:
le_risk = LabelEncoder()
y_clf = le_risk.fit_transform(y_clf)

Spliteamos.

In [17]:
X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X_clf,
    y_clf,
    test_size=0.2,
    random_state=42,
    stratify=y_clf
)

Escalamos las variables numéricas.

In [18]:
scaler_clf = StandardScaler()

X_train_clf[["O3", "PM2.5"]] = scaler_clf.fit_transform(X_train_clf[["O3", "PM2.5"]])
X_test_clf[["O3", "PM2.5"]] = scaler_clf.transform(X_test_clf[["O3", "PM2.5"]])


Ahora ya pasamos a la parte final, que es la predicción de la variable objetivo.

In [19]:

# Modelo 1: Decision Tree

model_clf_1 = DecisionTreeClassifier(random_state=42)
model_clf_1.fit(X_train_clf, y_train_clf)

pred_clf_1 = model_clf_1.predict(X_test_clf)
bal_acc_1 = balanced_accuracy_score(y_test_clf, pred_clf_1)

# Modelo 2: KNN

model_clf_2 = KNeighborsClassifier()
model_clf_2.fit(X_train_clf, y_train_clf)

pred_clf_2 = model_clf_2.predict(X_test_clf)
bal_acc_2 = balanced_accuracy_score(y_test_clf, pred_clf_2)

Resultados

In [20]:
print("\nMISIÓN 3 - Predicción de Environmental_risk")
print("Decision Tree -> Balanced Accuracy:", bal_acc_1)
print("KNN -> Balanced Accuracy:", bal_acc_2)

# elegir mejor modelo
if bal_acc_1 >= bal_acc_2:
    mejor_modelo_clf = model_clf_1
    print("Mejor modelo: Decision Tree")
else:
    mejor_modelo_clf = model_clf_2
    print("Mejor modelo: KNN")


MISIÓN 3 - Predicción de Environmental_risk
Decision Tree -> Balanced Accuracy: 0.9992236024844721
KNN -> Balanced Accuracy: 0.6238250517598344
Mejor modelo: Decision Tree


In [21]:
# # chequeo de sanidad
# y_train_random = np.random.permutation(y_train_clf)

# model_random = DecisionTreeClassifier(random_state=42)
# model_random.fit(X_train_clf, y_train_random)
# pred_random = model_random.predict(X_test_clf)
# acc_random = balanced_accuracy_score(y_test_clf, pred_random)

# print("\nBaseline aleatorio - Decision Tree -> Accuracy:", acc_random)

In [22]:
mask_fill_risk = (
    df["Environmental_risk"].isna() &
    df["Year"].notna() &
    df["Month"].notna() &
    df["Day"].notna() &
    df["O3"].notna() &
    df["PM2.5"].notna()
)

if mask_fill_risk.sum() > 0:
    X_fill = df.loc[mask_fill_risk, features_clf].copy()

    X_fill[["O3", "PM2.5"]] = scaler_clf.transform(X_fill[["O3", "PM2.5"]])

    pred_fill = mejor_modelo_clf.predict(X_fill)

    df.loc[mask_fill_risk, "Environmental_risk"] = le_risk.inverse_transform(pred_fill)

In [23]:
print("\nVALORES FALTANTES EN Environmental_risk DESPUÉS DE IMPUTACIÓN:", df["Environmental_risk"].isna().sum())


VALORES FALTANTES EN Environmental_risk DESPUÉS DE IMPUTACIÓN: 0
